# NexaTel Customer Churn: Phase 3 KPI Calculations

**What this notebook does:** it calculates the 10 KPIs from the PRD, each with its formula, its value and a short reading of the result. The tables the dashboard needs in Phase 4 are saved to `NexaTel/kpi/`.

## Decisions and assumptions (read these first)
- **Reporting window:** the last 12 months of the data, April 2025 to March 2026. The data ends on 31 March 2026.
- **Monthly churn:** for each month, *customers at start* are customers who joined before the month began and had not churned yet. *Customers lost* are those whose `churn_date` falls inside the month.
- **MRR** is the sum of the monthly charge of every subscription with status Active, as the PRD says. 251 of those subscriptions belong to customers who are marked Churned, worth about Rs.3.1 lakh a month. The notebook shows MRR and ARPU with and without them. A further 52 Active subscriptions start after 31 March 2026 and are flagged as well.
- **ARPU** is MRR divided by the number of active customers on 31 March 2026.
- **Gross margin is an assumption.** No NexaTel document gives one. CLV uses 40% and the notebook shows 30% and 50% as well. Treat CLV as a way to compare segments, not as a forecast.
- **Support and contract KPIs** use the tables as they are. The notes under each KPI say what is left out.
- This data gives lower monthly churn (0.4% to 1.3%) than the 1.9% to 2.7% in the PRD background text. All numbers here are calculated from `churn_date`.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import matplotlib.pyplot as plt

BASE = "/content/drive/MyDrive/NexaTel/"
DATA = BASE + "data/"
KPI_DIR = BASE + "kpi/"
os.makedirs(KPI_DIR, exist_ok=True)

customers = pd.read_csv(DATA + "customers_clean.csv",
                        parse_dates=["date_of_birth", "acquisition_date", "churn_date"])
subs = pd.read_csv(DATA + "subscriptions_clean.csv")
subs["start_date"] = pd.to_datetime(subs["start_date"], format="%d-%m-%Y", errors="coerce")
subs["end_date"]   = pd.to_datetime(subs["end_date"],   format="%d-%m-%Y", errors="coerce")
tickets   = pd.read_csv(DATA + "support_tickets.csv")
contracts = pd.read_csv(DATA + "contracts.csv")
contracts["end_date"] = pd.to_datetime(contracts["end_date"], format="%d-%m-%Y", errors="coerce")

customers["churned"] = customers["customer_status"] == "Churned"
months = pd.period_range("2025-04", "2026-03", freq="M")      # the 12 reporting months

print("customers:", customers.shape, "| subscriptions:", subs.shape,
      "| tickets:", tickets.shape, "| contracts:", contracts.shape)

## KPI 1 and KPI 2: Customer churn rate and customer retention rate
- **Churn rate** = Customers lost / Customers at start x 100 (target: below 2.1% a month)
- **Retention rate** = (Customers at end - New customers) / Customers at start x 100

In [ ]:
def monthly_table(df, months):
    rows = []
    for m in months:
        month_start, month_end = m.start_time, m.end_time
        # customers who were with us at the start of the month
        at_start = (df["acquisition_date"] < month_start) & (df["churn_date"].isnull() | (df["churn_date"] >= month_start))
        start = at_start.sum()
        lost = (at_start & (df["churn_date"] >= month_start) & (df["churn_date"] <= month_end)).sum()
        new = ((df["acquisition_date"] >= month_start) & (df["acquisition_date"] <= month_end)).sum()
        end = ((df["acquisition_date"] <= month_end) & (df["churn_date"].isnull() | (df["churn_date"] > month_end))).sum()
        rows.append({"month": str(m), "start": start, "lost": lost, "new": new, "end": end})
    t = pd.DataFrame(rows)
    t["churn_pct"] = (t["lost"] / t["start"] * 100).round(2)
    t["retention_pct"] = ((t["end"] - t["new"]) / t["start"] * 100).round(2)
    return t

monthly = monthly_table(customers, months)
print(monthly.to_string(index=False))

avg_churn = monthly["churn_pct"].mean()
avg_retention = monthly["retention_pct"].mean()
print(f"\nKPI 1  Average monthly churn rate (12 months): {avg_churn:.2f}%   | latest month ({monthly['month'].iloc[-1]}): {monthly['churn_pct'].iloc[-1]}%")
print(f"KPI 2  Average monthly retention rate (12 months): {avg_retention:.2f}%")

In [ ]:
# Monthly churn trend with the 2.1% target
plt.figure(figsize=(9, 4))
plt.plot(monthly["month"], monthly["churn_pct"], marker="o", color="#D1495B")
plt.axhline(2.1, color="black", linestyle="--", label="Target: below 2.1%")
plt.ylim(0, 2.5)
plt.title("Monthly churn rate, Apr 2025 to Mar 2026")
plt.xlabel("Month")
plt.ylabel("Churn rate (%)")
plt.xticks(rotation=45)
plt.legend()
plt.savefig(KPI_DIR + "kpi01_monthly_churn_trend.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading KPI 1 and 2:** Monthly churn averaged 0.65% over the last 12 months, well under the 2.1% target, and retention averaged 99.33%. Churn has trended upward, with a few small dips, from 0.44% in April 2025 to 1.25% in March 2026, so the target is met but the trend is the thing to watch. March is the highest month by a wide margin. Acquisition dates in the data stop on 1 March, so the newest month may be incomplete, and I cannot tell yet whether the March jump is real.

## KPI 6 and KPI 3: Monthly Recurring Revenue and ARPU
- **MRR** = sum of the monthly charge of all Active subscriptions
- **ARPU** = MRR / Active customers (on 31 March 2026)

In [ ]:
active_subs = subs[subs["subscription_status"] == "Active"].merge(
    customers[["customer_id", "churned", "customer_segment"]], on="customer_id", how="left")

mrr = active_subs["monthly_charge_inr"].sum()
active_customers = (~customers["churned"]).sum()
arpu = mrr / active_customers

# Same numbers without the 251 Active subscriptions that belong to customers marked Churned
odd = active_subs[active_subs["churned"]]
mrr_adj = active_subs[~active_subs["churned"]]["monthly_charge_inr"].sum()
arpu_adj = mrr_adj / active_customers

print(f"KPI 6  MRR: Rs.{mrr:,.0f}   (Active subscriptions: {len(active_subs):,})")
print(f"KPI 3  ARPU: Rs.{arpu:,.0f} per active customer   (Active customers: {active_customers:,})")
print(f"Note: {len(odd)} Active subscriptions belong to Churned customers, worth Rs.{odd['monthly_charge_inr'].sum():,.0f} a month")
print(f"Without them: MRR Rs.{mrr_adj:,.0f} | ARPU Rs.{arpu_adj:,.0f}")

# Active subscriptions whose start date is after the end of the data
future_start = active_subs[active_subs["start_date"] > "2026-03-31"]
print(f"Note: {len(future_start)} Active subscriptions start after 31 Mar 2026, worth Rs.{future_start['monthly_charge_inr'].sum():,.0f} a month")

**Reading KPI 6 and 3:** The recurring revenue base is about Rs.1.50 crore a month (Rs.15,031,009), which works out to Rs.970 per active customer. 251 Active subscriptions sit with customers marked Churned, worth Rs.3.1 lakh a month. This looks like a data inconsistency, and removing them lowers ARPU to Rs.950. A further 52 Active subscriptions (Rs.59,741 a month) have start dates after 31 March 2026. They are inside the MRR above, but not in the month-by-month MRR table saved for the dashboard, which counts a subscription only from its start date. That is why March in that table (Rs.14,971,268) is lower than the KPI 6 figure.

## KPI 5: Revenue lost to churn
Churned customers x their ARPU. Each churned customer has exactly one terminated subscription, so their monthly charge is their ARPU. This KPI covers customers who churned in the last 12 months.

In [ ]:
terminated = subs[subs["subscription_status"] == "Terminated"].merge(
    customers[["customer_id", "churn_date"]], on="customer_id", how="left")
lost_12m = terminated[(terminated["churn_date"] >= "2025-04-01") & (terminated["churn_date"] <= "2026-03-31")]

churned_12m = lost_12m["customer_id"].nunique()
rev_lost_monthly = lost_12m["monthly_charge_inr"].sum()
avg_per_churned = rev_lost_monthly / churned_12m

print(f"KPI 5  Customers churned in the last 12 months: {churned_12m:,}")
print(f"       Their average monthly charge: Rs.{avg_per_churned:,.1f}")
print(f"       Monthly recurring revenue lost: Rs.{rev_lost_monthly:,.0f}")
print(f"       Annualised (x12): Rs.{rev_lost_monthly * 12:,.0f}")
print(f"Check: {churned_12m:,} x Rs.{avg_per_churned:,.1f} = Rs.{churned_12m * avg_per_churned:,.0f}")

**Reading KPI 5:** 1,265 customers churned in the last 12 months and took Rs.774,988 of monthly recurring revenue with them (about Rs.613 each). At that run rate it is about Rs.93 lakh (Rs.9.3 million) of revenue a year.

## KPI 4: Customer Lifetime Value (CLV)
CLV = ARPU x Gross margin % x (1 / monthly churn rate).

**Gross margin is an assumption (40%).** It is not in any NexaTel document. The sensitivity table below shows how the answer moves with the margin.

In [ ]:
GROSS_MARGIN = 0.40          # ASSUMPTION, not from NexaTel data. Change it here and re-run.

avg_churn_decimal = avg_churn / 100
clv_overall = arpu * GROSS_MARGIN / avg_churn_decimal
print(f"Overall CLV at {GROSS_MARGIN:.0%} margin: Rs.{clv_overall:,.0f}  (implied customer life: {1 / avg_churn_decimal:,.0f} months)")

# Sensitivity: what if the margin is different?
print("\nOverall CLV at other margins:")
for gm in [0.30, 0.40, 0.50]:
    print(f"  {gm:.0%}: Rs.{arpu * gm / avg_churn_decimal:,.0f}")

In [ ]:
# CLV by segment: each segment gets its own ARPU and its own monthly churn
rows = []
for seg in ["Mass", "Value", "Premium", "Enterprise"]:
    seg_customers = customers[customers["customer_segment"] == seg]
    seg_churn = monthly_table(seg_customers, months)["churn_pct"].mean() / 100
    seg_mrr = active_subs[active_subs["customer_segment"] == seg]["monthly_charge_inr"].sum()
    seg_active = (~seg_customers["churned"]).sum()
    seg_arpu = seg_mrr / seg_active
    rows.append({"segment": seg, "active_customers": seg_active, "arpu_inr": round(seg_arpu),
                 "monthly_churn_%": round(seg_churn * 100, 2),
                 "implied_life_months": round(1 / seg_churn),
                 "clv_inr": round(seg_arpu * GROSS_MARGIN / seg_churn)})
clv_by_segment = pd.DataFrame(rows)
print(clv_by_segment.to_string(index=False))
print("\nPremium CLV is", round(clv_by_segment.loc[2, "clv_inr"] / clv_by_segment.loc[0, "clv_inr"], 1), "times Mass CLV")

**Reading KPI 4:** With the assumed 40% margin the overall CLV is Rs.60,007. By segment, CLV rises steeply: Mass Rs.14,186, Value Rs.27,547, Premium Rs.74,809 and Enterprise Rs.398,243. Premium is about 5.3 times Mass, as the PRD expects. Two cautions: the margin is an assumption, and because monthly churn is so low the formula implies customer lifetimes of 8 to 20 years (97 to 242 months). That is why the numbers are best read as a comparison between segments, not as a forecast.

## KPI 7 and KPI 8: First Contact Resolution and average resolution time
- **FCR** = tickets resolved on first contact / total tickets x 100 (target: above 78%)
- **Average resolution time** = mean resolution hours (target: below 24 hours). The ticket table already gives `resolution_hours`.

In [ ]:
total_tickets = len(tickets)
fcr_yes = (tickets["first_contact_resolution"] == "Yes").sum()
fcr = fcr_yes / total_tickets * 100

resolved = tickets[tickets["status"] == "Resolved"]
fcr_resolved_only = (resolved["first_contact_resolution"] == "Yes").mean() * 100
open_tickets = total_tickets - len(resolved)

avg_resolution = resolved["resolution_hours"].mean()
median_resolution = resolved["resolution_hours"].median()

print(f"KPI 7  First Contact Resolution: {fcr:.1f}%   ({fcr_yes:,} of {total_tickets:,} tickets)")
print(f"       FCR among resolved tickets only: {fcr_resolved_only:.1f}%")
print(f"KPI 8  Average resolution time: {avg_resolution:.1f} hours   (median {median_resolution:.1f})")
print(f"Tickets not resolved (Pending, Escalated, Reopened): {open_tickets:,} of {total_tickets:,} ({open_tickets / total_tickets:.1%})")

**Reading KPI 7 and 8:** First contact resolution is 40.4% (12,726 of 31,490 tickets), far below the 78% target. Part of that gap is the formula: 13,366 tickets are still Pending, Escalated or Reopened and can never count as resolved on first contact. Even among resolved tickets the rate is 70.2%, still under target. Resolution time looks fine at 16.0 hours on average (median 13.5), under the 24-hour target. But that average only includes the 18,124 resolved tickets, so the 42% of tickets still open, which are the slowest ones, are not in it.

## KPI 9: Contract renewal rate
Renewal rate = contracts renewed / contracts due for renewal x 100 (target: above 88% for postpaid and fibre).

**Definition used:** a contract is *due* if its end date is on or before 31 March 2026 and its status shows a decision: Renewed, Auto-Renewed or Not Renewed. Renewed means Renewed or Auto-Renewed.

In [ ]:
ended = contracts[contracts["end_date"] <= "2026-03-31"]
decided = ended[ended["renewal_status"].isin(["Renewed", "Auto-Renewed", "Not Renewed"])]
renewed = decided["renewal_status"].isin(["Renewed", "Auto-Renewed"])
renewal_rate = renewed.mean() * 100

pf = decided[decided["contract_type"].isin(["Postpaid Mobile", "Fiber Broadband"])]
renewal_rate_pf = pf["renewal_status"].isin(["Renewed", "Auto-Renewed"]).mean() * 100

left_out = ended[~ended["renewal_status"].isin(["Renewed", "Auto-Renewed", "Not Renewed"])]["renewal_status"].value_counts()

print(f"KPI 9  Contract renewal rate (all types): {renewal_rate:.1f}%   ({renewed.sum():,} of {len(decided):,} due)")
print(f"       Postpaid Mobile + Fiber Broadband only: {renewal_rate_pf:.1f}%   ({len(pf):,} contracts)")
print("\nContracts past their end date but left out of the rate:")
print(left_out.to_string())

**Reading KPI 9:** 86.7% of contracts that came due were renewed (5,516 of 6,362). For postpaid and fibre, the contract types the target applies to, the rate is 84.0%, below the 88% target. The rate leaves out 342 cancelled contracts and 2,947 contracts that are past their end date but still show as Active or Pending. Counting cancellations as "not renewed" would lower the overall rate to about 82%.

## KPI 10: Average customer tenure
Mean `tenure_months` of the active customer base.

In [ ]:
avg_tenure = customers[~customers["churned"]]["tenure_months"].mean()
print(f"KPI 10  Average tenure of active customers: {avg_tenure:.1f} months  ({avg_tenure / 12:.1f} years)")
print(f"        (All customers, including churned: {customers['tenure_months'].mean():.1f} months)")

**Reading KPI 10:** Active customers have been with NexaTel for 87.1 months on average, about 7.3 years. A single figure cannot show whether stickiness is improving, because that needs tenure compared across time. Treat it as a description of a mature base.

## KPI summary and files for the dashboard

In [ ]:
def status(met, has_target=True):
    return ("Met" if met else "Missed") if has_target else "No target"

summary = pd.DataFrame([
    ["1  Customer churn rate (monthly avg, 12 months)", f"{avg_churn:.2f}%", "below 2.1%", status(avg_churn < 2.1)],
    ["2  Customer retention rate (monthly avg)",        f"{avg_retention:.2f}%", "-", status(True, False)],
    ["3  ARPU",                                          f"Rs.{arpu:,.0f}", "-", status(True, False)],
    ["4  CLV (40% margin assumed)",                      f"Rs.{clv_overall:,.0f}", "-", status(True, False)],
    ["5  Revenue lost to churn (monthly, last 12 mo)",   f"Rs.{rev_lost_monthly:,.0f}", "-", status(True, False)],
    ["6  Monthly recurring revenue",                     f"Rs.{mrr:,.0f}", "-", status(True, False)],
    ["7  First contact resolution",                      f"{fcr:.1f}%", "above 78%", status(fcr > 78)],
    ["8  Average resolution time",                       f"{avg_resolution:.1f} hours", "below 24 hours", status(avg_resolution < 24)],
    ["9  Contract renewal rate (postpaid + fibre)",      f"{renewal_rate_pf:.1f}%", "above 88%", status(renewal_rate_pf > 88)],
    ["10 Average tenure (active customers)",             f"{avg_tenure:.1f} months", "-", status(True, False)],
], columns=["KPI", "Value", "Target", "Status"])
summary

In [ ]:
# Monthly table for the Phase 4 dashboard: churn, retention, MRR and revenue lost, one row per month
mrr_by_month, lost_by_month = [], []
for m in months:
    month_end = m.end_time
    live = subs[(subs["start_date"] <= month_end) & (subs["end_date"].isnull() | (subs["end_date"] > month_end))]
    mrr_by_month.append(live["monthly_charge_inr"].sum())
    ended_now = subs[(subs["subscription_status"] == "Terminated") & (subs["end_date"] >= m.start_time) & (subs["end_date"] <= month_end)]
    lost_by_month.append(ended_now["monthly_charge_inr"].sum())

monthly_kpis = monthly.copy()
monthly_kpis["mrr_inr"] = mrr_by_month
monthly_kpis["revenue_lost_inr"] = lost_by_month

monthly_kpis.to_csv(KPI_DIR + "monthly_kpis.csv", index=False)
summary.to_csv(KPI_DIR + "kpi_summary.csv", index=False)
clv_by_segment.to_csv(KPI_DIR + "clv_by_segment.csv", index=False)
print("Saved to", KPI_DIR)
monthly_kpis

## Notes for the report
- Every number above is calculated in this notebook. The gross margin (40%) is the only number not taken from the data.
- The KPIs that miss their target are first contact resolution (40.4% vs 78%) and postpaid/fibre contract renewal (84.0% vs 88%). Churn and resolution time meet theirs.
- Open data questions to mention: 251 Active subscriptions held by Churned customers, 52 Active subscriptions that start after 31 March 2026, 2,947 contracts past their end date still marked Active or Pending, and a March 2026 churn spike that may be partly a data cut-off effect.